# EEP 564 - TinyML - Assignment 2: Network Anomaly Detection

## Introduction

This assignment focuses on network anomaly detection using deep neural networks (DNNs) and model optimization techniques. We will work with the Network Anomaly Dataset, which contains 125,973 samples with 42 features representing normal and attack network connections. The main tasks involve data preprocessing, dimensionality reduction visualization, DNN implementation, dynamic range quantization, and Arduino deployment of the quantized model.

## Preparation

First of all, we will install required packages for data processing, machine learning, and model optimization in this homework:

In [ ]:
# Uninstall packages on Google Colab because new versions of TensorFlow has changed some TFLite APIs
#%pip uninstall -y tensorflow tensorflow-model-optimization

# a) Install with CPU only support:
%pip install tensorflow<2.21
# b) Alternatively, with CUDA (GPU) support on Linux:
#%pip install tensorflow[with-cuda]<2.21

%pip install numpy pandas scikit-learn tensorflow-model-optimization tf_keras tqdm

### Note: Resolving `keras.src` Namespace Issue

When using TensorFlow and TensorFlow Model Optimization in Colab, you may encounter a `keras.src` namespace issue, causing incompatibility with `tensorflow_model_optimization.quantization.keras`. To resolve this:

1. Set the `KERAS_BACKEND` environment variable to `tensorflow` before importing TensorFlow.
2. Ensure you are using compatible versions of TensorFlow (`>=2.12`) and TensorFlow Model Optimization.
3. Clone the model using `tensorflow.keras.models.clone_model()` to ensure it aligns with the `tensorflow.keras` namespace.
4. Always restart the runtime and reinstall TensorFlow-related packages to avoid lingering conflicts.

This ensures that all operations use the correct `tensorflow.keras` implementation, avoiding compatibility issues.

In [ ]:
import os

# Resolve keras.src namespace issue with TensorFlow Model Optimization
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## Question 1: Dataset Loading and Processing

We start by loading the Network Anomaly Dataset and preparaing it for preprocessing by adding the column titles for the 42 features:

In [ ]:
import pandas as pd

# Load the Network Anomaly Dataset with feature names
column_names = [
    "duration", "protocoltype", "service", "flag", "srcbytes", "dstbytes",
    "land", "wrongfragment", "urgent", "hot", "numfailedlogins", "loggedin",
    "numcompromised", "rootshell", "suattempted", "numroot", "numfilecreations",
    "numshells", "numaccessfiles", "numoutboundcmds", "ishostlogin", "isguestlogin",
    "count", "srvcount", "serrorrate", "srvserrorrate", "rerrorrate", "srvrerrorrate",
    "samesrvrate", "diffsrvrate", "srvdiffhostrate", "dsthostcount", "dsthostsrvcount",
    "dsthostsamesrvrate", "dsthostdiffsrvrate", "dsthostsamesrcportrate",
    "dsthostsrvdiffhostrate", "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate", "attack", "lastflag"
]

data = pd.read_csv("Network_anomaly_data.txt", sep=",", names=column_names)

Now, complete the following data preprocessing steps:

1) Drop the `land`, `urgent`, `numfailedlogins`, and `numoutboundcmds` columns from the `DataFrame`.
2) Replace any label that is not named `normal` to `attack` in the attack column.
3) Use `sklearn.preprocessing.LabelEncoder` to convert non-numerical attributes in `protocoltype`, `service`, `flag`, and `attack` columns to numerical values.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# [ TODO ]
# 1) Drop the specified columns from the dataframe
# 2) Standardize non-`normal` labels
# 3) Use LabelEncoder to convert categorical features to numerical values
raise NotImplementedError

## Question 2: Dimensionality Reduction for Visualization

Now that we have loaded and processed our dataset, it's time to visualize its data distribution with various techniques. In this part we will try three dimensionality reduction techniques (t-SNE, PCA, KernelPCA) we have already practiced in class. We will plot figures with red marks for attacks and blue marks for normal traffic.

### Task 2a: t-SNE Visualization

Use t-SNE from `sklearn.manifold.TSNE` to visualize **the test set** in 2D with class-based coloring.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# [ TODO ]
# Apply t-SNE on X_test and create a visualization
raise NotImplementedError

### Task 2b: PCA Visualization

Use PCA from `sklearn.decomposition.PCA` to visualize **the test set** in 2D with class-based coloring. Alternatively you can also use `sklearn.decomposition.IncrementalPCA` to speed up the visualization process.

In [ ]:
from sklearn.decomposition import PCA

# [TODO]
# Apply PCA on X_test and create a visualization


### Task 2c: KernelPCA Visualization

Use `KernelPCA` from `sklearn.decomposition.KernelPCA` with RBF kernel to visualize **the test set** in 2D with class-based coloring. You can randomly reduce the amount of data to visualize if this step takes too long, but you **must not** overwrite the original test set if you choose to do so.

In [ ]:
from sklearn.decomposition import KernelPCA

# [TODO]
# Apply KernelPCA with RBF kernel on X_test and create a visualization


## Question 3: Implementing a Deep Neural Network

Alright, it's time to build and train a deep neural network model for anomaly detection. We will create a DNN with an appropriate architecture to classify network traffic as normal or attack, using two output neurons with softmax activation to provide probability distributions over the two classes.

### Task 3a: Define DNN Architecture

To build our network anomaly detection model, we start by defining the architecture of a Sequential neural network. The model should include appropriate hidden layers and an output layer with 2 neurons using softmax activation for binary classification.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# [ TODO ]
# Define the neural network model below
base_model = Sequential([
    # Add a few (about 2 to 4) hidden layers here with appropriate number of neurons and activation functions
    # The last layer should have 2 neurons and `softmax` activation
    NotImplemented
])

### Task 3b: Compile and Train Model

Once the model architecture is defined, we compile it with an appropriate optimizer and loss function, then train it on our training data to learn patterns that distinguish between normal and attack network traffic.

In [ ]:
# [ TODO ]
# Compile and train the `base_model`` on the training set
# Use `categorical_crossentropy` loss, the `Adam` optimizer with appropriate learning rate,
# and train for an appropriate number of epochs
raise NotImplementedError

### Task 3c: Evaluate Model Performance

After training, we evaluate the model's performance on **the unseen test set** using classification metrics to understand how well our model generalizes to new network traffic data.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# [ TODO ]
# Evaluate `base_model` on test set and print the `classification_report` and `confusion_matrix`
raise NotImplementedError

When all of these steps are done, we save our model weights in file:

In [ ]:
# Save the Keras model weights to a HDF5 file
base_model.save("original-model.h5")

## Question 4: Dynamic Range Quantization

Okay, our model is now ready, but we still need to perform a few optimization steps for deployment on resource-constrained devices like our Arduino boards. In this part we will use Dynamic Range Quantzation, which significantly reduces model size while maintaining reasonable accuracy, making deployment feasible on embedded hardware.

### Task 4a: Apply Dynamic Range Quantization

We convert our trained Keras model to TensorFlow Lite (TFLite) format with dynamic range quantization. This process reduces the model's memory footprint significantly, making it suitable for embedded deployment on microcontrollers with limited storage and computational resources.

In [ ]:
# Load the trained model
base_model = tf.keras.models.load_model("original-model.h5")

# [ TODO ]
# Apply dynamic range quantization to convert the model to TFLite format
tflite_quant_model = NotImplemented

In [ ]:
# Save the quantized model
with open("quantized-model.tflite", 'wb') as f:
    f.write(tflite_quant_model)

# Get the file sizes
original_model_size = os.path.getsize("original-model.h5")
quantized_model_size = os.path.getsize("quantized-model.tflite")

# Print the model sizes
print(f"Original model size: {original_model_size / 1024:.2f} KB")
print(f"Quantized model size: {quantized_model_size / 1024:.2f} KB")

### Task 4b: Evaluate Quantized Model

We evaluate the quantized model on the test set to measure the impact of quantization on model accuracy and compare it with the original floating-point model. This helps us understand the trade-off between model compression and prediction performance.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# [ TODO ]
# Evaluate `tflite_quant_model` on test set and print the `classification_report` and `confusion_matrix`
raise NotImplementedError

## Converting TFLite Model to C Header

To deploy our quantized model on the Arduino Nano BLE, we need to convert it to a C header file. This allows the microcontroller to load the model directly into its memory and run inference without external dependencies.

In [ ]:
# Import c_writer utility for model conversion
import c_writer

# c_writer is a py file in the same folder and has been imported at the beginning of the notebook
# Reference : https://github.com/ShawnHymel/tinyml-example-anomaly-detection/blob/master/utils/c_writer.py
# We use #04x to pad the output to 2 digits with a 0x prefix
hex_array = [format(val, '#04x') for val in tflite_quant_model]
# Calling function to convert an array into a C string (requires Numpy)
# create_array(np_array, var_type, var_name, line_limit=80, indent=4)
c_model = c_writer.create_array(np.array(hex_array), 'unsigned char', "network_model")
# Calling Function to create a header file with given C code as a string
header_str = c_writer.create_header(c_model, "network_model")


In [ ]:
# Write the header file to disk
with open("network_model.h", "w") as f:
    f.write(header_str)

## Generating Test Samples for Arduino Inference

In this final step, we extract test samples and convert them to C-formatted arrays so they can be embedded directly in the Arduino sketch for on-device inference testing.

In [ ]:
# Extract and convert the first 5 test samples (indices 0-4)
Xtest_first5 = X_test[:5, :]
print(c_writer.create_array(Xtest_first5, "float", "X_test"))

In [ ]:
# Extract and convert the corresponding labels for the first 5 test samples
ytest_first5 = y_test[:5]
print(c_writer.create_array(ytest_first5, "uint8_t", "y_test"))